In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES']='2'

In [6]:
from scunet import SCUNet
import torch
from torchsummary import summary
import numpy as np

In [3]:
model = SCUNet( 
        in_nc=1, 
        config=[2,2,2,2,2,2,2], 
        dim=32, 
        drop_path_rate=0.0, 
        input_resolution=48, 
        head_dim=16, 
        window_size=3,
        )

Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000


In [5]:
# SUMMARIZE MODEL
model.children

<bound method Module.children of SCUNet(
  (m_head): Sequential(
    (0): Conv3d(1, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
  )
  (m_down1): Sequential(
    (0): ConvTransBlock(
      (trans_block): Block(
        (ln1): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
        (msa): WMSA(
          (embedding_layer): Linear(in_features=16, out_features=48, bias=True)
          (linear): Linear(in_features=16, out_features=16, bias=True)
        )
        (drop_path): Identity()
        (ln2): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
        (mlp): Sequential(
          (0): Linear(in_features=16, out_features=64, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=64, out_features=16, bias=True)
        )
      )
      (conv1_1): Conv3d(32, 32, kernel_size=(1, 1, 1), stride=(1, 1, 1))
      (conv1_2): Conv3d(32, 32, kernel_size=(1, 1, 1), stride=(1, 1, 1))
      (conv_block): Sequential(
        (0): 

In [6]:
# a way to create a custom dataset
# 

In [7]:
from torch.utils.data import Dataset
class SimpleDataSet(Dataset):
    def __init__(self) -> None:
        super().__init__()

    def __getitem__(self, idx):
        x = np.random.rand(size=(1,48,48,48), dtype=np.float32)
        y = np.random.randint(0,2, size=(1,48,48,48), dtype=np.float32)
        return x, y

In [17]:
from torch.utils.data import Dataset, DataLoader
import json 
from einops import rearrange 

class SimpleDataSet(Dataset):

    def __init__(self, file_path):
        super().__init__()
        self.filename = json.load(open(file_path, 'r'))

    def __getitem__(self, idx):
        xy_pair = self.filename[idx]
        x = np.load(xy_pair[0])
        x = rearrange(x, 'h w l c -> c h w l')
        y = np.load(xy_pair[1])
        y = rearrange(y, 'h w l c -> c h w l')
        return x, y
    
    def __len__(self):
        return len(self.filename)
    

file_path = '/home/tnw-nb4020-03/bep_lotte_micelle/training_cubes/scunet_cz48/cubedata_directory/cubedata_validation/XY_filenames_dataset.json'
dataset = SimpleDataSet(file_path)

train_dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [8]:
type(train_dataloader)
print(next(iter(train_dataloader))[1].shape)

torch.Size([1, 1, 48, 48, 48])


In [12]:
cube1 = next(iter(train_dataloader))[0]
print(cube1.shape)
print(type(cube1))

model.eval()
pred1 = model(cube1)

print(cube1.shape, pred1.shape)

torch.Size([1, 1, 48, 48, 48])
<class 'torch.Tensor'>
torch.Size([1, 1, 48, 48, 48]) torch.Size([1, 1, 48, 48, 48])


In [16]:
model.training

True

In [15]:
model.train()


SCUNet(
  (m_head): Sequential(
    (0): Conv3d(1, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
  )
  (m_down1): Sequential(
    (0): ConvTransBlock(
      (trans_block): Block(
        (ln1): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
        (msa): WMSA(
          (embedding_layer): Linear(in_features=16, out_features=48, bias=True)
          (linear): Linear(in_features=16, out_features=16, bias=True)
        )
        (drop_path): Identity()
        (ln2): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
        (mlp): Sequential(
          (0): Linear(in_features=16, out_features=64, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=64, out_features=16, bias=True)
        )
      )
      (conv1_1): Conv3d(32, 32, kernel_size=(1, 1, 1), stride=(1, 1, 1))
      (conv1_2): Conv3d(32, 32, kernel_size=(1, 1, 1), stride=(1, 1, 1))
      (conv_block): Sequential(
        (0): Conv3d(16, 16, kernel_size=(3, 3,

In [10]:
type(dataset)
print(type(dataset[0][0]))

<class 'numpy.ndarray'>


In [11]:
data   = torch.rand(1,1,48,48,48)
labels = torch.rand(1,1,48,48,48)
out = model(data)
print(data.shape)
print(labels.shape)

torch.Size([1, 1, 48, 48, 48])
torch.Size([1, 1, 48, 48, 48])


In [10]:
## GENERATING RANDOM INPUT DATA
from torch.utils.data import DataLoader, TensorDataset
import numpy as np


num_samples = 1
# data  = [np.random.rand(1, 48, 48, 48).astype(np.float32) for _ in range(num_samples)]
# masks = [np.random.randint(0, 2, size=(1, 48, 48, 48)).astype(np.int64) for _ in range(num_samples)]

# data  = np.array(data)
# masks = np.array(masks)

# data_ts  = torch.tensor(data)
# masks_ts = torch.tensor(masks)

# data_ts  = torch.rand(size=(num_samples, 1, 48, 48, 48), dtype=np.float32)
# masks_ts = torch.randint(0, 2, (num_samples, 1, 48, 48, 48), dtype=np.float32)

# (num_batches, num_channels, img_heigth, img_length, img_width)
dataset = [(np.random.rand(1, 48, 48, 48).astype(np.float32), np.random.randint(0, 2, (1, 48, 48, 48)).astype(np.float32)) for i in range(num_samples)]

# dataset = TensorDataset(data_ts, masks_ts)
# dataset = [(data_ts[i], masks_ts[i]) for i in range(num_samples)]

batch_size = 1
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [22]:
## TRAINING
from torch import nn
import torch.nn.functional as F
from torch.optim import Adam
from tqdm import tqdm
optimizer = Adam(model.parameters(), lr=0.01)
# loss_fn   = nn.MSELoss()
loss_fn   = nn.BCEWithLogitsLoss()
smax      = nn.Softmax()
epochs    = 10
batches   = 1
num_samples = batches

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = model.to(device)

for epoch in range(epochs):
    running_loss = 0
    for image, mask in tqdm(train_dataloader, desc=f'Epoch {epoch + 1}/{epochs}'):
        # Transfer data to GPU if available
        image = image.to(device)
        mask  = mask.to(device)
        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        output = model(image)
        output = torch.tanh(output)
        # output = smax(output)
        # output = F.softmax(output.view(48*48*48), dim=0).view(batches, 1, 48, 48, 48) # convert to 1D, apply softmax, and reshape
        # # > should resolve this error:
        # UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
        
        # Calculate the loss
        loss = loss_fn(output, mask)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        # Print statistics
        running_loss += loss.item()

    # Print average loss for the epoch
    epoch_loss = running_loss / len(train_dataloader)
    print(f"Epoch [{epoch + 1}/{epochs}], Loss: {epoch_loss:.4f}")


Epoch 1/10:   0%|          | 0/14128 [00:00<?, ?it/s]

Epoch 1/10:   1%|▏         | 196/14128 [00:52<1:02:46,  3.70it/s]


KeyboardInterrupt: 

: 

In [ ]:
from skorch import NeuralNetClassifier

net = NeuralNetClassifier(SCUNet,
                          batch_size=1,
                          criterion= nn.BCEWithLogitsLoss,
                          )


In [ ]:
mydata = SimpleDataSet()
net.fit(mydata, y=None)

Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.015385
Block Initial Type: W, drop_path_rate:0.030769
Block Initial Type: SW, drop_path_rate:0.046154
Block Initial Type: W, drop_path_rate:0.061538
Block Initial Type: SW, drop_path_rate:0.076923
Block Initial Type: W, drop_path_rate:0.092308
Block Initial Type: SW, drop_path_rate:0.107692
Block Initial Type: W, drop_path_rate:0.123077
Block Initial Type: SW, drop_path_rate:0.138462
Block Initial Type: W, drop_path_rate:0.153846
Block Initial Type: SW, drop_path_rate:0.169231
Block Initial Type: W, drop_path_rate:0.184615
Block Initial Type: SW, drop_path_rate:0.200000


ValueError: Stratified CV requires explicitly passing a suitable y.

In [ ]:
l = loss_fn(mask, output)

In [ ]:
o.shape

NameError: name 'o' is not defined